# 04 — QLoRA ile Instruction Fine-Tuning

**Kapsam:** Transformer tabanlı büyük dil modelleri üzerinde eğitim; LoRA, QLoRA, PEFT
ve instruction tuning teknikleri.

## Neden QLoRA?
- **Full fine-tuning**: Tüm parametreleri günceller. Milyarlarca parametre için Colab'ın
  ücretsiz T4 GPU'suna (~15GB VRAM) sığmaz.
- **LoRA**: Temel modeli dondurur, her katmana küçük "adaptör" matrisleri ekler; sadece
  bunları eğitir (parametrelerin <%1'i).
- **QLoRA**: LoRA'nın üstüne, temel modeli 4-bit'e (NF4) sıkıştırarak bellek ihtiyacını
  daha da azaltır — böylece 3B-7B modeller tek bir T4'e sığar.

Bu notebook, 01. notebook'ta ürettiğiniz `sft_dataset.jsonl` (taslak notlar → düzgün
doküman çiftleri) ile devam eder.

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) unpack adımını atlar.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
#   4) EN ÖNEMLİSİ: data/, models/, mlruns/ klasörlerini Drive'daki kalıcı bir
#      klasöre sembolik bağlantı (symlink) yapar. Neden gerekli: /content her
#      runtime'da sıfırlanır, yani 01. notebook'ta ürettiğiniz corpus.jsonl gibi
#      dosyalar farklı bir runtime'da (örn. 02. notebook'u açtığınızda) KAYBOLUR.
#      Bu adım olmadan her notebook'u ayrı ayrı çalıştırdığınızda önceki adımların
#      ürettiği veriyi bulamazsınız. Sembolik bağlantı sayesinde hangi runtime'da
#      olursanız olun aynı kalıcı depoyu okur/yazarsınız.
import os, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"
DRIVE_DATA_DIR = "/content/drive/MyDrive/baykar-nlp-hazirlik-data"
PERSIST_DIRS = ["data", "models", "mlruns"]

try:
    from google.colab import drive
    # force_remount=True KULLANMIYORUZ: bu, zaten mount edilmişken bile her seferinde
    # yeniden yetkilendirme (izin penceresi) ister, gereksiz bekleme/kesinti yaratır.
    # drive.mount() zaten mount edilmişse kendi içinde anında geri döner; mount
    # edilmemişse (bu runtime'da ilk çalıştırma) normal şekilde izin ister — bu
    # durumda çıkan izin penceresini/bağlantısını tamamlamanız gerekir, hücreyi
    # durdurmayın.
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False  # Colab dışında (yerelde) çalışıyorsanız Drive adımları atlanır.

if not os.path.exists(PROJECT_DIR) and IN_COLAB:
    if os.path.exists(DRIVE_ZIP_PATH):
        import shutil
        shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
    else:
        print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
              "yükleyin ya da kendi reponuzu klonlayın: "
              f"!git clone <repo-url> {PROJECT_DIR}")

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

if IN_COLAB and os.path.exists(PROJECT_DIR):
    import shutil
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for _name in PERSIST_DIRS:
        _drive_path = os.path.join(DRIVE_DATA_DIR, _name)
        os.makedirs(_drive_path, exist_ok=True)
        _local_path = os.path.join(PROJECT_DIR, _name)

        if os.path.islink(_local_path):
            continue  # zaten Drive'a bağlanmış

        if os.path.isdir(_local_path):
            # Zip'ten gelen boş klasörü kaldırıp yerine symlink koyuyoruz. İçinde
            # (nadiren) veri varsa önce Drive'a taşıyoruz, hiçbir şeyi kaybetmiyoruz.
            for _item in os.listdir(_local_path):
                _src = os.path.join(_local_path, _item)
                _dst = os.path.join(_drive_path, _item)
                if not os.path.exists(_dst):
                    shutil.move(_src, _dst)
            shutil.rmtree(_local_path)

        os.symlink(_drive_path, _local_path)

    print("Kalıcı veri klasörü:", DRIVE_DATA_DIR)


## Colab ortam düzeltmeleri

Kurulum hücresinden hemen sonra çalıştırın. Paket sürümlerini sabitler ve Transformers v5 uyumluluk yamalarını `src/` üzerine yazar (eski zip ile gelseniz bile).

In [ ]:
import os, subprocess, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
bootstrap = os.path.join(PROJECT_DIR, "scripts", "colab_bootstrap.py")
qlora_fix = os.path.join(PROJECT_DIR, "scripts", "colab_qlora_fix.py")

if os.path.exists(bootstrap):
    subprocess.run([sys.executable, bootstrap], check=True, cwd=PROJECT_DIR)
elif os.path.exists(qlora_fix):
    subprocess.run([sys.executable, qlora_fix], check=True, cwd=PROJECT_DIR)
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "pyarrow==17.0.0", "datasets==2.21.0"],
            check=True,
        )
    except subprocess.CalledProcessError:
        pass
else:
    raise FileNotFoundError(
        "Guncel zip gerekli (scripts/colab_bootstrap.py veya colab_qlora_fix.py). "
        "Asagidaki 'qlora yama' hucreesini calistirin."
    )

### Eski zip fallback (colab_patches yoksa)

Bootstrap hücresi `FileNotFoundError` verirse **bu hücreyi** çalıştırın. `qlora_train.py` yamasını doğrudan yazar; ayrı zip güncellemesi gerekmez.

In [ ]:
# colab_patches GEREKTIRMEZ — eski zip ile calisir
import os, subprocess

PROJECT = "/content/baykar-nlp-hazirlik"
fix_script = os.path.join(PROJECT, "scripts", "colab_qlora_fix.py")
dst = os.path.join(PROJECT, "src/finetune/qlora_train.py")

if os.path.exists(fix_script):
    subprocess.run([__import__("sys").executable, fix_script], check=True, cwd=PROJECT)
else:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    QLORA_SRC = r'''"""QLoRA fine-tuning."""
from __future__ import annotations
import inspect
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from src.config import MODEL_CONFIG, QLORA_CONFIG
from src.finetune.dataset_utils import build_hf_dataset

def load_quantized_model_and_tokenizer(base_model=None):
    base_model = base_model or MODEL_CONFIG.base_llm
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(base_model)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(base_model, quantization_config=bnb, device_map="auto")
    return prepare_model_for_kbit_training(m), tok

def build_lora_model(model):
    cfg = LoraConfig(r=QLORA_CONFIG.lora_r, lora_alpha=QLORA_CONFIG.lora_alpha,
        lora_dropout=QLORA_CONFIG.lora_dropout, target_modules=list(QLORA_CONFIG.target_modules),
        bias="none", task_type="CAUSAL_LM")
    p = get_peft_model(model, cfg); p.print_trainable_parameters(); return p

def _seq_length_config_kwargs():
    ml = QLORA_CONFIG.max_seq_length; p = inspect.signature(SFTConfig.__init__).parameters
    if "max_length" in p: return {"max_length": ml}
    if "max_seq_length" in p: return {"max_seq_length": ml}
    return {}

def _seq_length_trainer_kwargs():
    if _seq_length_config_kwargs(): return {}
    ml = QLORA_CONFIG.max_seq_length; p = inspect.signature(SFTTrainer.__init__).parameters
    if "max_seq_length" in p: return {"max_seq_length": ml}
    if "max_length" in p: return {"max_length": ml}
    return {}

def train(base_model=None, output_dir=None):
    output_dir = output_dir or QLORA_CONFIG.output_dir
    model, tok = load_quantized_model_and_tokenizer(base_model)
    peft = build_lora_model(model)
    tr, ev = build_hf_dataset()
    kw = dict(output_dir=output_dir, num_train_epochs=QLORA_CONFIG.num_train_epochs,
        per_device_train_batch_size=QLORA_CONFIG.per_device_train_batch_size,
        gradient_accumulation_steps=QLORA_CONFIG.gradient_accumulation_steps,
        learning_rate=QLORA_CONFIG.learning_rate, logging_steps=10,
        eval_strategy="epoch", save_strategy="epoch", bf16=True, report_to=["mlflow"])
    kw.update(_seq_length_config_kwargs())
    cp = inspect.signature(SFTConfig.__init__).parameters
    if "assistant_only_loss" in cp: kw["assistant_only_loss"] = True
    cfg = SFTConfig(**kw)
    tp = inspect.signature(SFTTrainer.__init__).parameters
    tk = dict(model=peft, args=cfg, train_dataset=tr, eval_dataset=ev)
    tk.update(_seq_length_trainer_kwargs())
    tk["processing_class" if "processing_class" in tp else "tokenizer"] = tok
    t = SFTTrainer(**tk); t.train(); t.save_model(output_dir); tok.save_pretrained(output_dir)
    return output_dir
'''
    open(dst, "w", encoding="utf-8").write(QLORA_SRC)

assert "_seq_length_config_kwargs" in open(dst, encoding="utf-8").read()
print("qlora_train.py guncellendi OK")

In [ ]:
import torch
print("CUDA kullanılabilir mi:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("UYARI: GPU bulunamadı. Runtime > Değiştir çalışma zamanı türü > T4 GPU seçin.")


## Hiperparametreler

`src/config.py` içindeki `QLoraConfig`'i inceleyin: `lora_r`, `lora_alpha`, `target_modules` gibi değerler burada tanımlı.

In [ ]:
from src.config import QLORA_CONFIG
print(QLORA_CONFIG)


## Eğitim

Bu hücre GPU'da birkaç dakika ile birkaç saat arasında sürebilir (veri seti boyutuna bağlı).

In [ ]:
from src.finetune.qlora_train import train

adapter_path = train()
print("LoRA adaptörü kaydedildi ->", adapter_path)


## Hızlı doğrulama

Eğitilen adaptörle RAG pipeline'ını tekrar çalıştırıp, temel modelle karşılaştırın (bkz. 07. notebook A/B test).

In [ ]:
from src.rag.rag_pipeline import answer

result = answer("Bir sorun giderme rehberi nasıl yapılandırılmalı?", model_path=adapter_path)
print(result["answer"])
